In [1]:
import importlib.util
from pathlib import Path
import numpy as np
import torch
import py3Dmol
from ase.io import read, write
from ase.build import molecule
from ase.cluster import Icosahedron
from torch.utils.data import DataLoader

# ==============================================================================
# 1. 環境設定とモデルファイルの自動ダウンロード
# ==============================================================================
WORKDIR = Path.cwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "s128_t64.nequip.ckpt"
MODEL_FILE_ID = "13qNINxBcToLHlWKY5OmKsnXpsdbzzQbk"
ckpt_path = WORKDIR / MODEL_NAME

if not ckpt_path.exists():
    import gdown
    print(f"Downloading model to {ckpt_path} ...")
    gdown.download(f"https://drive.google.com/uc?id={MODEL_FILE_ID}", str(ckpt_path), quiet=False)

# ==============================================================================
# 2. モデルパラメータの抽出
# ==============================================================================
checkpoint = torch.load(str(ckpt_path), map_location=device, weights_only=False)
model_config = checkpoint["hyper_parameters"]["model"]
inner_model_config = model_config.get("model", {})

avgn = inner_model_config.get("avg_num_neighbors", 1.0)
sgij = 1.0 / np.sqrt(2.0 * avgn)
escal = inner_model_config.get("per_type_energy_scales", 1.0)
if np.all(np.asarray(escal) == 0.0):
    escal = 1.0

# ==============================================================================
# 3. モデルおよびZBLモジュールのロード
# ==============================================================================
from nequip.utils.global_state import set_global_state
from nequip.scripts.compile import load_saved_model, _EAGER_MODEL_KEY, _SOLE_MODEL_KEY
from nequip.nn import graph_model
from nequip.nn.pair_potential import ZBL
from nequip.data import AtomicDataDict
from nequip.data.dataset import ASEDataset
from nequip.data.transforms import NeighborListTransform, ChemicalSpeciesToAtomTypeMapper

set_global_state(allow_tf32=True)
model = load_saved_model(str(ckpt_path), _EAGER_MODEL_KEY, _SOLE_MODEL_KEY).to(device)
model.eval()

type_names = model.metadata[graph_model.TYPE_NAMES_KEY].split(" ")
r_max = float(model.metadata[graph_model.R_MAX_KEY])

zbl_module = ZBL(
    type_names,
    type_names,
    units="metal",
    irreps_in={AtomicDataDict.NORM_LENGTH_KEY: "0e"},
).to(device)

print(f"Ready: Model loaded ({device}) | r_max = {r_max} Å")

# ==============================================================================
# 4. 解析関数群の定義
# ==============================================================================
def make_dataloader_for_xyz(xyz_path, chemical_symbols, r_max, device):
    """XYZファイルから推論用DataLoaderを作成"""
    transforms = [
        NeighborListTransform(r_max=r_max),
        ChemicalSpeciesToAtomTypeMapper(chemical_symbols),
    ]
    dataset = ASEDataset(str(xyz_path), transforms=transforms)
    dl = DataLoader(dataset, batch_size=1, collate_fn=AtomicDataDict.batched_from_list)

    class DeviceDataLoader:
        def __init__(self, dl, device):
            self.dl = dl
            self.device = device
        def __iter__(self):
            for batch in self.dl:
                yield {k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        def __len__(self):
            return len(self.dl)

    return DeviceDataLoader(dl, device)


def calculate_dij_table(out, chemical_symbols, cutoff=5.0):
    """元のチュートリアルと全く同じレイアウト・集計項目で D_ij テーブルを表示"""
    DKEY = AtomicDataDict
    
    # テンソルから多重次元を取り除いて確実に1次元化
    edge_eng = out[DKEY.EDGE_ENERGY_KEY].detach().cpu().numpy().reshape(-1)
    edge_idx = out[DKEY.EDGE_INDEX_KEY].detach().cpu().numpy()
    if edge_idx.ndim == 2 and edge_idx.shape[0] == 2:
        edge_idx = edge_idx.T
    edge_r = out[DKEY.EDGE_LENGTH_KEY].detach().cpu().numpy().reshape(-1)
    types = out[DKEY.ATOM_TYPE_KEY].detach().cpu().numpy().reshape(-1)

    # ZBL項の計算
    edge_eng_zbl = None
    zbl_key = getattr(DKEY, "EDGE_ENERGY_ZBL_KEY", None)
    if zbl_key is not None and zbl_key in out:
        edge_eng_zbl = out[zbl_key].detach().cpu().numpy().reshape(-1)
    else:
        for try_key in ("edge_energy_zbl", "EDGE_ENERGY_ZBL_KEY", "edge_energy_zbl_key"):
            if try_key in out:
                edge_eng_zbl = out[try_key].detach().cpu().numpy().reshape(-1)
                break

    if edge_eng_zbl is None and zbl_module is not None:
        try:
            zbl_out = zbl_module._zbl(
                Z=zbl_module.atomic_numbers,
                r=out[DKEY.EDGE_LENGTH_KEY].view(-1),
                atom_types=out[DKEY.ATOM_TYPE_KEY],
                edge_index=out[DKEY.EDGE_INDEX_KEY],
                qqr2exesquare=zbl_module._qqr2exesquare,
            )
            norm_k = getattr(DKEY, "NORM_LENGTH_KEY", DKEY.EDGE_LENGTH_KEY)
            zbl_cutoff = zbl_module.cutoff(out[norm_k]).to(torch.get_default_dtype())
            edge_eng_zbl = (zbl_out * zbl_cutoff).detach().cpu().numpy().reshape(-1)
        except Exception:
            edge_eng_zbl = np.zeros_like(edge_eng)

    if edge_eng_zbl is None:
        edge_eng_zbl = np.zeros_like(edge_eng)

    # 元のノートブックと同一のヘッダー書式
    print("idx_i idx_j  sym_i-sym_j      r (Å)    D_ij (eV)")
    print("-" * 55)

    sumint = 0.0
    sumdij = 0.0
    sumzbl = 0.0
    cnt = 0
    rec = {}

    for e_idx, eng, eng_z, r in zip(edge_idx, edge_eng, edge_eng_zbl, edge_r):
        r_val = float(np.asarray(r).squeeze())
        if r_val > cutoff:
            continue

        i_idx = int(e_idx[0])
        j_idx = int(e_idx[1])

        key = (i_idx, j_idx, round(r_val, 6))
        rev = (j_idx, i_idx, round(r_val, 6))

        val_eng = float(np.asarray(eng).squeeze())
        val_eng_z = float(np.asarray(eng_z).squeeze())

        if rev in rec:
            other = rec.pop(rev)

            # Core項（双方向の合算 × スケール係数）
            dij_core = (val_eng + other["eng"]) * sgij * escal
            dij_core = float(np.asarray(dij_core).squeeze())

            # ZBL項
            zbl_pair = 2.0 * val_eng_z
            dij_total = dij_core + zbl_pair

            sumdij += dij_core
            sumzbl += zbl_pair
            sumint += dij_total
            cnt += 1

            sym_i = chemical_symbols[int(types[i_idx])]
            sym_j = chemical_symbols[int(types[j_idx])]

            # 元の出力と同一のフォーマット: "  1-  0    C-C         1.486      -4.3108"
            print(
                f"{i_idx:3d}-{j_idx:3d}   "
                f"{sym_i:>2s}-{sym_j:<2s}      "
                f"{r_val:7.3f}   {dij_total:10.4f}"
            )
        else:
            rec[key] = {"eng": val_eng, "eng_zbl": val_eng_z}

    # 元の出力と同一のサマリー項目
    print("-" * 55)
    print(f"Number of pairs : {cnt}")
    print(f"Sum D_ij        : {sumint:.6f} eV")
    print(f"Core term sum   : {sumdij:.6f} eV")
    print(f"ZBL term sum    : {sumzbl:.6f} eV")

    if rec:
        print(f"Warning: {len(rec)} directed edges did not find a reverse partner.")


def inspect_and_analyze(structure_input, name="sample", cutoff=None, show_3d=True):
    cutoff = cutoff or r_max
    if isinstance(structure_input, (str, Path)):
        atoms = read(str(structure_input))
        temp_xyz = WORKDIR / f"temp_{Path(structure_input).stem}.xyz"
    else:
        atoms = structure_input.copy()
        temp_xyz = WORKDIR / f"temp_{name}.xyz"

    # 結晶などの周期境界がある場合、孤立クラスターとして安全に書き出すため PBC をオフにする
    atoms.set_pbc(False)
    write(str(temp_xyz), atoms)

    if show_3d:
        with open(temp_xyz) as f:
            xyz_str = f.read()
        view = py3Dmol.view(width=600, height=350)
        view.addModel(xyz_str, "xyz")
        view.setStyle({"stick": {"radius": 0.15}, "sphere": {"scale": 0.28}})
        
        # 原子数が多すぎる（35個以上）場合はラベル表示をスキップして動作を軽くする
        if len(atoms) <= 35:
            for idx, (sym, pos) in enumerate(zip(atoms.get_chemical_symbols(), atoms.positions)):
                view.addLabel(
                    f"{sym}{idx}",
                    {"position": {"x": float(pos[0]), "y": float(pos[1]), "z": float(pos[2])},
                     "fontSize": 11, "fontColor": "black", "backgroundColor": "white", "backgroundOpacity": 0.7, "inFront": True}
                )
        else:
            print(f"※ 原子数が多いため（{len(atoms)} 個）、3Dラベルの描画を省略しました。")
            
        view.zoomTo()
        view.show()

    dl = make_dataloader_for_xyz(temp_xyz, type_names, r_max, device)
    for batch in dl:
        out = model(batch)
        print(f"Target: {name} (Formula: {atoms.get_chemical_formula()}, Atoms: {len(atoms)})")
        calculate_dij_table(out, type_names, cutoff=cutoff)
        return out, atoms


def show_pair_energy(out, atom_i, atom_j, chemical_symbols, zbl_module=None):
    """特定の原子ペア (atom_i, atom_j) の D_ij と距離を表示する"""
    DKEY = AtomicDataDict

    atom_i = int(atom_i)
    atom_j = int(atom_j)

    if atom_i == atom_j:
        raise ValueError("異なる原子IDを指定してください。")

    edge_eng = out[DKEY.EDGE_ENERGY_KEY].detach().cpu().numpy()
    edge_idx = out[DKEY.EDGE_INDEX_KEY].detach().cpu().numpy().squeeze()
    if edge_idx.ndim == 2 and edge_idx.shape[0] == 2:
        edge_idx = edge_idx.T

    edge_r = out[DKEY.EDGE_LENGTH_KEY].detach().cpu().numpy().reshape(-1)
    types = out[DKEY.ATOM_TYPE_KEY].detach().cpu().numpy().reshape(-1)

    n_atoms = len(types)
    if not (0 <= atom_i < n_atoms and 0 <= atom_j < n_atoms):
        raise IndexError(f"原子IDは 0 から {n_atoms - 1} の範囲で指定してください。")

    # ZBL 成分の取得
    edge_eng_zbl = None
    zbl_key = getattr(DKEY, "EDGE_ENERGY_ZBL_KEY", None)
    if zbl_key is not None and zbl_key in out:
        edge_eng_zbl = out[zbl_key].detach().cpu().numpy()
    else:
        for try_key in ("edge_energy_zbl", "EDGE_ENERGY_ZBL_KEY", "edge_energy_zbl_key"):
            if try_key in out:
                edge_eng_zbl = out[try_key].detach().cpu().numpy()
                break

    if edge_eng_zbl is None and zbl_module is not None:
        try:
            zbl_out = zbl_module._zbl(
                Z=zbl_module.atomic_numbers,
                r=out[DKEY.EDGE_LENGTH_KEY].view(-1),
                atom_types=out[DKEY.ATOM_TYPE_KEY],
                edge_index=out[DKEY.EDGE_INDEX_KEY],
                qqr2exesquare=zbl_module._qqr2exesquare,
            )
            edge_eng_zbl = zbl_out.detach().cpu().numpy()
            norm_k = getattr(DKEY, "NORM_LENGTH_KEY", DKEY.EDGE_LENGTH_KEY)
            zbl_edge_cutoff = zbl_module.cutoff(out[norm_k]).to(torch.get_default_dtype()).detach().cpu().numpy()
            edge_eng_zbl = edge_eng_zbl * zbl_edge_cutoff
        except Exception:
            edge_eng_zbl = np.zeros_like(edge_eng)

    if edge_eng_zbl is None:
        edge_eng_zbl = np.zeros_like(edge_eng)

    # 有向エッジ (i -> j) と (j -> i) を探索
    idx_ij = None
    idx_ji = None
    for edge_no, (i, j) in enumerate(edge_idx):
        i, j = int(i), int(j)
        if i == atom_i and j == atom_j:
            idx_ij = edge_no
        elif i == atom_j and j == atom_i:
            idx_ji = edge_no

    sym_i = chemical_symbols[int(types[atom_i])]
    sym_j = chemical_symbols[int(types[atom_j])]

    # カットオフ外のペアの距離計算（KeyError を防ぐため atoms オブジェクトを参照）
    if idx_ij is None or idx_ji is None:
        if "atoms" in globals():
            pos_i = atoms.positions[atom_i]
            pos_j = atoms.positions[atom_j]
            distance = float(np.linalg.norm(pos_i - pos_j))
        else:
            distance = float("nan")

        print("Selected atomic pair")
        print("--------------------")
        print(f"Atoms    : {sym_i}{atom_i} - {sym_j}{atom_j}")
        print(f"Distance : {distance:.4f} Å")
        print(f"D_ij     : not available (カットオフ r_max = {r_max:.3f} Å の範囲外です)")
        return

    # E3D ペアエネルギーの合算 (双方向成分 + ZBL)
    eng_ij = float(np.asarray(edge_eng[idx_ij]).sum())
    eng_ji = float(np.asarray(edge_eng[idx_ji]).sum())
    zbl_ij = float(np.asarray(edge_eng_zbl[idx_ij]).sum())
    zbl_ji = float(np.asarray(edge_eng_zbl[idx_ji]).sum())

    dij_core = (eng_ij + eng_ji) * sgij * escal
    dij_total = float(dij_core + (zbl_ij + zbl_ji))
    distance = float(np.asarray(edge_r[idx_ij]).squeeze())

    print("Selected atomic pair")
    print("--------------------")
    print(f"Atoms    : {sym_i}{atom_i} - {sym_j}{atom_j}")
    print(f"Distance : {distance:.4f} Å")
    print(f"D_ij     : {dij_total:.6f} eV")

/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-10-05 11:44:16,565][nequip.model.saved_models.load_utils][INFO] - [rank: 0] Loading model from /home/mamem/research/interatomic-bond-analyzer/s128_t64.nequip.ckpt ...
[2026-10-05 11:44:16,568][nequip.model.saved_models.checkpoint][INFO] - [rank: 0] Loading model from checkpoint file: /home/mamem/research/interatomic-bond-analyzer/s128_t64.nequip.ckpt ...


/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/nequip/model/saved_models/checkpoint.py:69: UserWarning: `torch` versions differ between the checkpoint file (2.7.1) and the current run (2.14.1+cpu) -- `ModelFromCheckpoint` will be built with the current run's versions, but please check that this decision is as intended.
  warnings.warn(
/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/nequip/model/saved_models/checkpoint.py:69: UserWarning: `e3nn` versions differ between the checkpoint file (0.5.6) and the current run (0.6.0) -- `ModelFromCheckpoint` will be built with the current run's versions, but please check that this decision is as intended.
  warnings.warn(
/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/nequip/model/saved_models/checkpoint.py:69: UserWarning: `nequip` versions differ between the checkpoint file (0.14.0) and the current run (0.19.1) -- `ModelFromCheckpoint` will be built with the current run's v

[2026-10-05 11:44:18,483][nequip.train.ema][INFO] - [rank: 0] Loading EMA weights for evaluation model.
Ready: Model loaded (cpu) | r_max = 6.0 Å


In [2]:
# ------------------------------------------------------------------------------
# パターン 1: 有機分子・気体（ASE内蔵の162種など）
# ------------------------------------------------------------------------------
# target_structure = molecule("C6H6")
# target_name = "Benzene"
# target_cutoff = None  # None の場合はモデルのデフォルト (6.0 Å)

# ------------------------------------------------------------------------------
# パターン 2: 金属ナノクラスター
# ------------------------------------------------------------------------------
# target_structure = Icosahedron("Pt", noshells=2)
# target_name = "Pt13_Cluster"
# target_cutoff = 3.5

# ------------------------------------------------------------------------------
# パターン 3: 結晶構造ファイル (CIF, POSCAR など)
# ------------------------------------------------------------------------------
file_path = "TiO2.cif"         # 作業フォルダ直下に置いたファイル名
repeat_dim = (2, 2, 2)         # (1, 1, 1)なら単位格子、(2, 2, 2)ならスーパーセル

# # 結晶ファイルの読み込み
crystal_raw = read(file_path)
 
if repeat_dim != (1, 1, 1):
     target_structure = crystal_raw.repeat(repeat_dim)
     target_name = f"{Path(file_path).stem}_{repeat_dim[0]}x{repeat_dim[1]}x{repeat_dim[2]}"
else:
     target_structure = crystal_raw
     target_name = Path(file_path).stem
 
target_cutoff = 3.0

# ==============================================================================
# 🚀 実行部（どのパターンでも共通して残す）
# ==============================================================================
out, atoms = inspect_and_analyze(
    target_structure, 
    name=target_name, 
    cutoff=target_cutoff
)

※ 原子数が多いため（96 個）、3Dラベルの描画を省略しました。


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Target: TiO2_2x2x2 (Formula: O64Ti32, Atoms: 96)
idx_i idx_j  sym_i-sym_j      r (Å)    D_ij (eV)
-------------------------------------------------------
  4-  0    O-Ti        1.936      -6.3756
  4-  3    O-Ti        1.992      -6.2348
  5-  0    O-Ti        1.992      -5.2313
  5-  1    O-Ti        1.936      -6.0804
  6-  0    O-Ti        1.936      -6.3742
  6-  4    O-O         2.799      -0.1531
  6-  1    O-Ti        1.992      -6.1966
  6-  5    O-O         2.464      -0.2006
  7-  1    O-Ti        1.936      -6.8032
  7-  5    O-O         2.799      -0.1479
  8-  1    O-Ti        1.992      -5.4325
  8-  7    O-O         2.464      -0.2206
  9-  2    O-Ti        1.992      -7.0273
  9-  3    O-Ti        1.936      -6.8331
 10-  2    O-Ti        1.936      -6.7361
 10-  3    O-Ti        1.992      -5.3810
 10-  9    O-O         2.464      -0.2440
 11-  0    O-Ti        1.992      -5.2440
 11-  3    O-Ti        1.936      -6.1067
 11-  4    O-O         2.464      -0.2034
 11-  

In [3]:
# ==============================================================================
# 🔍 汎用：中心原子・第1配位圏（Coordination Shell）自動探索＆D_ij一括集計セル
# ==============================================================================
import numpy as np

# ------------------------------------------------------------------------------
# ⚙️ 設定パラメータ（対象物質に合わせて変更）
# ------------------------------------------------------------------------------
center_element = "Ti"     # 着目する中心元素（例: "Ti", "Zn", "Na", "Pt" など）
neighbor_element = "O"     # 結合相手の元素（例: "O", "S", "Cl" など。None にすると全元素を対象）
max_bond_length = 2.6      # 第1近接結合距離の上限 [Å]（通常の結合長 + 0.3〜0.5 Å 程度に設定）

# ------------------------------------------------------------------------------
# 🚀 探索・計算部（変更不要）
# ------------------------------------------------------------------------------
if "out" not in globals() or "atoms" not in globals() or out is None:
    raise RuntimeError("直前の実験設定セルを実行して 'out' と 'atoms' を取得してください。")

symbols = np.array(atoms.get_chemical_symbols())
positions = atoms.positions
n_total = len(atoms)

# 1. 構造全体の幾何学的重心（幾何中心）を算出
center_pos = positions.mean(axis=0)

# 2. 指定した中心元素の中で、最もブロックの中心に近い原子を特定
target_indices = np.where(symbols == center_element)[0]
if len(target_indices) == 0:
    raise ValueError(f"元素 '{center_element}' は現在の構造（全 {n_total} 原子）に存在しません。")

dists_to_geom_center = np.linalg.norm(positions[target_indices] - center_pos, axis=1)
center_idx = target_indices[np.argmin(dists_to_geom_center)]

print("=" * 68)
print(f"🎯 選択された中心原子 : {center_element}{center_idx} (全 {n_total} 原子中)")
print(f"   座標              : [{positions[center_idx][0]:.3f}, {positions[center_idx][1]:.3f}, {positions[center_idx][2]:.3f}] Å")
print("=" * 68)

# 3. 中心原子から max_bond_length 以内にある配位原子（第1配位圏）を探索
diffs = positions - positions[center_idx]
dists = np.linalg.norm(diffs, axis=1)

coord_candidates = []
for idx in range(n_total):
    if idx == center_idx:
        continue
    r = dists[idx]
    if r <= max_bond_length:
        sym = symbols[idx]
        if neighbor_element is None or sym == neighbor_element:
            coord_candidates.append((idx, sym, r))

# 距離順（近い順）にソート
coord_candidates.sort(key=lambda x: x[2])
coord_num = len(coord_candidates)

target_desc = f"{neighbor_element} 原子" if neighbor_element else "全配位原子"
print(f"配位数（CN）: {coord_num} 個の {target_desc} が {max_bond_length:.2f} Å 以内に存在します\n")

if coord_num == 0:
    print(f"⚠️ 指定された上限距離 ({max_bond_length} Å) 内に配位原子が見つかりませんでした。")
    print("   'max_bond_length' の値を少し大きくしてください。")
else:
    # 4. 各配位ペアの D_ij を一括取得・表示
    DKEY = AtomicDataDict
    edge_eng = out[DKEY.EDGE_ENERGY_KEY].detach().cpu().numpy().reshape(-1)
    edge_idx = out[DKEY.EDGE_INDEX_KEY].detach().cpu().numpy()
    if edge_idx.ndim == 2 and edge_idx.shape[0] == 2:
        edge_idx = edge_idx.T
    edge_r = out[DKEY.EDGE_LENGTH_KEY].detach().cpu().numpy().reshape(-1)

    edge_eng_zbl = None
    zbl_key = getattr(DKEY, "EDGE_ENERGY_ZBL_KEY", None)
    if zbl_key is not None and zbl_key in out:
        edge_eng_zbl = out[zbl_key].detach().cpu().numpy().reshape(-1)
    else:
        for try_key in ("edge_energy_zbl", "EDGE_ENERGY_ZBL_KEY", "edge_energy_zbl_key"):
            if try_key in out:
                edge_eng_zbl = out[try_key].detach().cpu().numpy().reshape(-1)
                break

    if edge_eng_zbl is None and zbl_module is not None:
        try:
            zbl_out = zbl_module._zbl(
                Z=zbl_module.atomic_numbers,
                r=out[DKEY.EDGE_LENGTH_KEY].view(-1),
                atom_types=out[DKEY.ATOM_TYPE_KEY],
                edge_index=out[DKEY.EDGE_INDEX_KEY],
                qqr2exesquare=zbl_module._qqr2exesquare,
            )
            norm_k = getattr(DKEY, "NORM_LENGTH_KEY", DKEY.EDGE_LENGTH_KEY)
            zbl_cutoff = zbl_module.cutoff(out[norm_k]).to(torch.get_default_dtype())
            edge_eng_zbl = (zbl_out.detach().cpu().numpy() * zbl_cutoff.detach().cpu().numpy()).reshape(-1)
        except Exception:
            edge_eng_zbl = np.zeros_like(edge_eng)

    if edge_eng_zbl is None:
        edge_eng_zbl = np.zeros_like(edge_eng)

    # ペア探索用インデックス辞書
    pair_to_edge = {}
    for e_no, (i, j) in enumerate(edge_idx):
        pair_to_edge[(int(i), int(j))] = e_no

    print(f"{'No.':<4} {'原子ペア':^12} {'距離 r (Å)':^12} {'D_ij (eV)':^14} {'Core (eV)':^12} {'ZBL (eV)':^10}")
    print("-" * 68)

    sum_dij_shell = 0.0
    sum_core_shell = 0.0
    sum_zbl_shell = 0.0

    for rank, (j_idx, sym_j, r_val) in enumerate(coord_candidates, 1):
        idx_ij = pair_to_edge.get((center_idx, j_idx))
        idx_ji = pair_to_edge.get((j_idx, center_idx))

        if idx_ij is None or idx_ji is None:
            print(f"{rank:<4d} {center_element}{center_idx}-{sym_j}{j_idx:<3d}   {r_val:8.3f} Å   [カットオフ外またはエッジ未定義]")
            continue

        val_eng_ij = float(np.asarray(edge_eng[idx_ij]).squeeze())
        val_eng_ji = float(np.asarray(edge_eng[idx_ji]).squeeze())
        val_zbl_ij = float(np.asarray(edge_eng_zbl[idx_ij]).squeeze())

        dij_core = (val_eng_ij + val_eng_ji) * sgij * escal
        dij_zbl = 2.0 * val_zbl_ij
        dij_total = float(dij_core + dij_zbl)

        sum_dij_shell += dij_total
        sum_core_shell += dij_core
        sum_zbl_shell += dij_zbl

        print(f"{rank:<4d} {center_element}{center_idx}-{sym_j}{j_idx:<3d}   {r_val:8.3f}      {dij_total:10.4f}     {dij_core:10.4f}   {dij_zbl:8.4f}")

    print("-" * 68)
    print(f"配位結合本数 (CN)       : {coord_num} 本")
    print(f"1結合あたりの平均 D_ij    : {sum_dij_shell / coord_num:.4f} eV")
    print(f"中心原子の総結合エネルギー : {sum_dij_shell:.4f} eV (Core: {sum_core_shell:.4f} eV, ZBL: {sum_zbl_shell:.4f} eV)")

🎯 選択された中心原子 : Ti86 (全 96 原子中)
   座標              : [3.783, 3.783, 9.615] Å
配位数（CN）: 6 個の O 原子 が 2.60 Å 以内に存在します

No.      原子ペア       距離 r (Å)     D_ij (eV)     Core (eV)    ZBL (eV) 
--------------------------------------------------------------------
1    Ti86-O32       1.936         -5.8832        -5.9442     0.0609
2    Ti86-O80       1.936         -6.3739        -6.4348     0.0609
3    Ti86-O70       1.936         -5.8832        -5.9442     0.0609
4    Ti86-O94       1.936         -6.3739        -6.4348     0.0609
5    Ti86-O93       1.992         -5.2309        -5.2836     0.0528
6    Ti86-O79       1.992         -5.2309        -5.2836     0.0528
--------------------------------------------------------------------
配位結合本数 (CN)       : 6 本
1結合あたりの平均 D_ij    : -5.8293 eV
中心原子の総結合エネルギー : -34.9759 eV (Core: -35.3252 eV, ZBL: 0.3493 eV)


In [19]:
# ==============================================================================
# 🔍 特定ペア解析セル（Selected Pair Inspection）
# ==============================================================================
# 上の3Dビューアーで確認した原子ラベル（例: C0, O1 など）の番号（0始まり）を指定します。
# ==============================================================================

# 調べたい2つの原子のインデックスを指定
atom_i = 15
atom_j = 9


# ------------------------------------------------------------------------------
# 🚀 実行部（変更不要）
# ------------------------------------------------------------------------------
if "out" not in globals() or "atoms" not in globals() or out is None:
    raise RuntimeError("直前の実験設定セルを実行して 'out' と 'atoms' を取得してください。")

n_atoms = len(atoms)

# 指定インデックスが有効範囲内かチェック
if not (0 <= atom_i < n_atoms and 0 <= atom_j < n_atoms):
    print(f"⚠️ 指定されたインデックス ({atom_i}, {atom_j}) は無効です。")
    print(f"   現在の構造 '{target_name}' の原子数は {n_atoms} 個（指定可能: 0 〜 {n_atoms - 1}）です。")
elif atom_i == atom_j:
    print(f"⚠️ 同一の原子 ({atom_i}, {atom_j}) が指定されています。異なる原子を指定してください。")
else:
    sym_i = atoms.get_chemical_symbols()[atom_i]
    sym_j = atoms.get_chemical_symbols()[atom_j]
    
    print("=" * 60)
    print(f" 🎯 対象ペア: {sym_i}{atom_i} - {sym_j}{atom_j}  (系: {target_name})")
    print("=" * 60)
    
    # チュートリアル由来の個別ペア詳細関数を実行
    show_pair_energy(
        out,
        atom_i=atom_i,
        atom_j=atom_j,
        chemical_symbols=type_names,
        zbl_module=zbl_module,
    )

 🎯 対象ペア: S15 - Ge9  (系: Ag2GeS3)
Selected atomic pair
--------------------
Atoms    : S15 - Ge9
Distance : 2.1779 Å
D_ij     : 90.734641 eV
